In [ ]:

import os
import json
import random
import torch
import numpy as np
from torch.utils.data import Dataset
from PIL import Image
from transformers import (
    DetrImageProcessor,
    DetrForObjectDetection,
    Trainer,
    TrainingArguments
)
from google.colab import drive
drive.mount('/content/drive')

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed()

BASE_DIR = "/content/drive/MyDrive/Hard Hat Workers"
TRAIN_DIR = os.path.join(BASE_DIR, "train")
TRAIN_ANN = os.path.join(TRAIN_DIR, "annotations.json")
TEST_DIR = os.path.join(BASE_DIR, "test")
TEST_ANN = os.path.join(TEST_DIR, "annotations.json")

KEEP_CATEGORY_IDS = {1, 2}
TARGET_SIZE = (416, 416)

class FilteredCocoDataset(Dataset):
    def __init__(self, image_folder, annotation_file, processor, keep_cat_ids, target_size=TARGET_SIZE):
        self.image_folder = image_folder
        self.processor = processor
        self.keep_cat_ids = keep_cat_ids
        self.target_size = target_size

        with open(annotation_file, 'r') as f:
            coco_data = json.load(f)

        self.images = {img['id']: img for img in coco_data['images']}
        self.annotations = {}
        for ann in coco_data['annotations']:
            if ann['category_id'] in keep_cat_ids:
                x, y, w, h = ann['bbox']
                if w <= 0 or h <= 0:
                    continue
                img_id = ann['image_id']
                self.annotations.setdefault(img_id, []).append(ann)

        self.image_ids = sorted(list(self.annotations.keys()))

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        img_info = self.images[image_id]
        image_path = os.path.join(self.image_folder, img_info['file_name'])
        image = Image.open(image_path).convert("RGB")
        orig_w, orig_h = image.size

        image = image.resize(self.target_size, Image.BILINEAR)
        scale_x = self.target_size[0] / orig_w
        scale_y = self.target_size[1] / orig_h

        anns = self.annotations[image_id]
        labels = []
        for ann in anns:
            x, y, w, h = ann['bbox']
            new_x = x * scale_x
            new_y = y * scale_y
            new_w = w * scale_x
            new_h = h * scale_y
            if new_w <= 0 or new_h <= 0:
                continue
            new_x = max(0, min(new_x, self.target_size[0] - 1))
            new_y = max(0, min(new_y, self.target_size[1] - 1))
            new_w = min(new_w, self.target_size[0] - new_x)
            new_h = min(new_h, self.target_size[1] - new_y)
            labels.append({
                "category_id": ann["category_id"],
                "bbox": [new_x, new_y, new_w, new_h],
                "area": ann.get("area", new_w * new_h),
                "iscrowd": ann.get("iscrowd", 0),
                "image_id": image_id,
            })

        if not labels:

            return self.__getitem__((idx + 1) % len(self.image_ids))
        encoding = self.processor(
            images=image,
            annotations={"image_id": image_id, "annotations": labels},
            return_tensors="pt"
        )

        return {
            "pixel_values": encoding["pixel_values"].squeeze(0),
            "pixel_mask": encoding["pixel_mask"].squeeze(0),
            "labels": encoding["labels"][0],
        }

def collate_fn(batch):
    pixel_values = torch.stack([item["pixel_values"] for item in batch])
    pixel_mask = torch.stack([item["pixel_mask"] for item in batch])
    labels = [item["labels"] for item in batch]
    return {
        "pixel_values": pixel_values,
        "pixel_mask": pixel_mask,
        "labels": labels,
    }

model_name = "facebook/detr-resnet-50"
processor = DetrImageProcessor.from_pretrained(
    model_name,
    size={"height": 416, "width": 416},
    do_resize=False,

model = DetrForObjectDetection.from_pretrained(
    model_name,
    ignore_mismatched_sizes=True,
    num_labels=len(KEEP_CATEGORY_IDS) + 1,
    id2label={0: "background", 1: "head", 2: "helmet"},
    label2id={"background": 0, "head": 1, "helmet": 2},
)

train_dataset = FilteredCocoDataset(TRAIN_DIR, TRAIN_ANN, processor, KEEP_CATEGORY_IDS)
val_dataset = FilteredCocoDataset(TEST_DIR, TEST_ANN, processor, KEEP_CATEGORY_IDS)

print(f"Train images: {len(train_dataset)}")
print(f"Val images: {len(val_dataset)}")

training_args = TrainingArguments(
    output_dir="./detr_head_helmet",
    num_train_epochs=50,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=50,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    weight_decay=1e-4,
    max_grad_norm=0.1,
    remove_unused_columns=False,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collate_fn,
    processing_class=processor,
)

trainer.train()

trainer.save_model("./detr_head_helmet_final")
processor.save_pretrained("./detr_head_helmet_final")
print("Модель сохранена в ./detr_head_helmet_final")